# Phase 02: ETH Wheat Sequence Processing Colab Notebook
This notebook runs the full ETH wheat sequence processing pipeline on real or synthetic data: parsing metadata and symptom masks, computing pinned-denominator disease severity, generating sequence pairs for forecasting, building leak-free splits, and exporting EDA visualizations.

In [ ]:
# Step 1: Install required dependencies
!pip install -q pyarrow pillow pyyaml pydantic pandas scipy scikit-learn matplotlib tqdm requests

In [ ]:
# Step 2: Configure Workspace & Python Import Paths
import os
import sys
from pathlib import Path

# Set DATA_ROOT to Google Drive or local temp storage
if os.path.exists('/content/drive'):
    drive_root = '/content/drive/MyDrive/crop_prognosis_data'
    os.makedirs(drive_root, exist_ok=True)
    os.environ['DATA_ROOT'] = drive_root
else:
    os.environ['DATA_ROOT'] = '/content/data'
print('DATA_ROOT set to:', os.environ['DATA_ROOT'])

# Check for existing git repository root
candidates = [
    os.getcwd(),
    '/content/PlantDiseaseSeverity',
    '/content/crop-prognosis'
]

found_repo = False
for path in candidates:
    if os.path.exists(os.path.join(path, '.git')) and os.path.exists(os.path.join(path, 'ml')):
        os.chdir(path)
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"Using project repository root at: {path}")
        found_repo = True
        break

if not found_repo:
    print("Standalone Colab mode: Writing updated 'ml' package files to /content/ml...")
    os.makedirs('/content/ml/utils', exist_ok=True)
    os.makedirs('/content/ml/data', exist_ok=True)
    os.makedirs('/content/ml/configs', exist_ok=True)
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

    Path('/content/ml/__init__.py').write_text('')
    Path('/content/ml/utils/__init__.py').write_text('')
    Path('/content/ml/data/__init__.py').write_text('')

    # Write paths.py
    Path('/content/ml/utils/paths.py').write_text('''import os\nfrom pathlib import Path\nfrom typing import NamedTuple\n\nclass ProjectPaths(NamedTuple):\n    project_root: Path\n    data_root: Path\n    raw_data: Path\n    interim_data: Path\n    processed_data: Path\n    models_dir: Path\n    reports_dir: Path\n    figures_dir: Path\n    tables_dir: Path\n    configs_dir: Path\n    docs_dir: Path\n\ndef get_paths() -> ProjectPaths:\n    data_env = os.getenv("DATA_ROOT", "/content/data")\n    data_root = Path(data_env)\n    root = Path("/content")\n    raw_data = data_root / "raw"\n    interim_data = data_root / "interim"\n    processed_data = data_root / "processed"\n    reports_dir = root / "reports"\n    figures_dir = reports_dir / "figures"\n    tables_dir = reports_dir / "tables"\n    for p in [data_root, raw_data, interim_data, processed_data, reports_dir, figures_dir, tables_dir]:\n        p.mkdir(parents=True, exist_ok=True)\n    return ProjectPaths(\n        project_root=root,\n        data_root=data_root,\n        raw_data=raw_data,\n        interim_data=interim_data,\n        processed_data=processed_data,\n        models_dir=root / "models",\n        reports_dir=reports_dir,\n        figures_dir=figures_dir,\n        tables_dir=tables_dir,\n        configs_dir=root / "ml" / "configs",\n        docs_dir=root / "docs"\n    )\n''')

    # Write logging_utils.py
    Path('/content/ml/utils/logging_utils.py').write_text('''import logging\n\ndef get_logger(name: str) -> logging.Logger:\n    logger = logging.getLogger(name)\n    if not logger.handlers:\n        handler = logging.StreamHandler()\n        formatter = logging.Formatter('[%(asctime)s] %(levelname)s %(name)s: %(message)s')\n        handler.setFormatter(formatter)\n        logger.addHandler(handler)\n        logger.setLevel(logging.INFO)\n    return logger\n''')

    # Write io.py
    Path('/content/ml/utils/io.py').write_text('''import json\nimport pandas as pd\nfrom pathlib import Path\n\ndef write_parquet(df: pd.DataFrame, path: Path) -> Path:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_parquet(path, index=False)\n    return path\n\ndef read_parquet(path: Path) -> pd.DataFrame:\n    return pd.read_parquet(path)\n\ndef write_json(data: dict, path: Path) -> Path:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path, 'w') as f:\n        json.dump(data, f, indent=2)\n    return path\n''')

    # Write ml/data/eth_wheat.py
    with open('/content/ml/data/eth_wheat.py', 'w') as f:
        f.write('''import sys, re, argparse\nfrom pathlib import Path\nfrom typing import Any, Optional, Tuple\nimport numpy as np, pandas as pd\nfrom PIL import Image\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import write_parquet\nlogger = get_logger(__name__)\n\ndef parse_image_id(filename_stem: str) -> dict:\n    pattern = r"^(\\d{8}_\\d{6})_([A-Z0-9]+)_(L\\d+)$"\n    match = re.match(pattern, filename_stem)\n    if match:\n        ts_str, plot_id, leaf_no = match.groups()\n        timestamp = pd.to_datetime(ts_str, format="%Y%m%d_%H%M%S")\n        sample_id = f"{plot_id}_{leaf_no}"\n        return {"image_id": filename_stem, "timestamp": timestamp, "plot_id": plot_id, "leaf_no": leaf_no, "sample_id": sample_id}\n    parts = filename_stem.split("_")\n    sample_id = f"{parts[2] if len(parts)>2 else 'P0'}_{parts[3] if len(parts)>3 else 'L1'}"\n    return {"image_id": filename_stem, "timestamp": pd.Timestamp.now(), "plot_id": parts[2] if len(parts)>2 else "P0", "leaf_no": parts[3] if len(parts)>3 else "L1", "sample_id": sample_id}\n\ndef compute_mask_metrics(mask_array: np.ndarray) -> dict:\n    unique_vals, counts = np.unique(mask_array, return_counts=True)\n    val_map = dict(zip(unique_vals, counts))\n    healthy_cnt = val_map.get(1, 0)\n    chlorosis_cnt = val_map.get(2, 0)\n    necrosis_cnt = val_map.get(3, 0)\n    pustule_cnt = val_map.get(4, 0)\n    diseased_pixels = necrosis_cnt + pustule_cnt\n    leaf_pixels = healthy_cnt + chlorosis_cnt + diseased_pixels\n    diseased_mask = (mask_array == 3) | (mask_array == 4)\n    n_lesions, mean_lesion_area = _count_connected_lesions(diseased_mask)\n    return {"diseased_pixels": int(diseased_pixels), "leaf_pixels_frame": int(leaf_pixels), "necrosis_pixels": int(necrosis_cnt), "pustule_pixels": int(pustule_cnt), "n_lesions": int(n_lesions), "mean_lesion_area": float(mean_lesion_area)}\n\ndef _count_connected_lesions(binary_mask: np.ndarray) -> Tuple[int, float]:\n    if not np.any(binary_mask): return 0, 0.0\n    try:\n        from scipy.ndimage import label\n        labeled, num_features = label(binary_mask)\n        if num_features == 0: return 0, 0.0\n        sizes = [np.sum(labeled == i) for i in range(1, num_features + 1)]\n        return num_features, float(np.mean(sizes))\n    except Exception:\n        total = np.sum(binary_mask)\n        return (1 if total > 0 else 0), float(total)\n\ndef generate_synthetic_eth_dataset(raw_dir: Path) -> None:\n    meta_dir = raw_dir / "meta"\n    ts_dir = raw_dir / "processed" / "ts"\n    meta_dir.mkdir(parents=True, exist_ok=True)\n    ts_dir.mkdir(parents=True, exist_ok=True)\n    exp_design = []\n    cultivars = ["Arina", "CH_Claro", "Titan", "Runal", "Forel"]\n    treatments = ["FI2", "0FI1", "F0I", "0F0I"]\n    for i in range(1, 11):\n        plot_id = f"ESWW{i:03d}"\n        exp_design.append({"plot_id": plot_id, "cultivar": cultivars[(i - 1) % 5], "treatment": treatments[(i - 1) % 4], "batch": 1 if i <= 5 else 2, "year": 2023, "heading_date": "2023-05-15"})\n    pd.DataFrame(exp_design).to_csv(meta_dir / "experimental_design.csv", index=False)\n    dates = pd.date_range("2023-05-01", "2023-06-30", freq="h")\n    np.random.seed(42)\n    weather_data = [{"timestamp": d.strftime("%Y-%m-%d %H:%M:%S"), "temperature": 15.0 + 5.0 * np.sin(d.hour / 24.0 * 2 * np.pi) + np.random.normal(0, 0.5), "relative_humidity": 75.0 + 15.0 * np.cos(d.hour / 24.0 * 2 * np.pi) + np.random.normal(0, 1.0)} for d in dates]\n    pd.DataFrame(weather_data).to_csv(meta_dir / "weather_station_hourly.csv", index=False)\n    for i in range(1, 6):\n        plot_id = f"ESWW{i:03d}"\n        sample_id = f"{plot_id}_L1"\n        sample_dir = ts_dir / sample_id\n        sample_dir.mkdir(parents=True, exist_ok=True)\n        start_time = pd.Timestamp("2023-05-10 10:00:00")\n        for step in range(5):\n            ts = start_time + pd.Timedelta(days=step * 2)\n            ts_str = ts.strftime("%Y%m%d_%H%M%S")\n            img_id = f"{ts_str}_{plot_id}_L1"\n            mask = np.ones((100, 100), dtype=np.uint8)\n            lesion_size = 5 + step * 8\n            mask[10:10+lesion_size, 10:10+lesion_size] = 3\n            mask[0:2, 0:2] = 5\n            Image.fromarray(mask).save(sample_dir / f"{img_id}_mask.png")\n\ndef process_eth_dataset(raw_dir: Optional[Path] = None, processed_dir: Optional[Path] = None) -> dict:\n    paths = get_paths()\n    if raw_dir is None: raw_dir = paths.raw_data / "eth_wheat"\n    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "eth"\n    processed_dir.mkdir(parents=True, exist_ok=True)\n    meta_dir = raw_dir / "meta"\n    ts_dir = raw_dir / "processed" / "ts"\n    if not meta_dir.exists() or not (meta_dir / "experimental_design.csv").exists():\n        generate_synthetic_eth_dataset(raw_dir)\n    exp_df = pd.read_csv(meta_dir / "experimental_design.csv")\n    weather_csv = meta_dir / "weather_station_hourly.csv"\n    if weather_csv.exists():\n        weather_h = pd.read_csv(weather_csv)\n        weather_h["timestamp"] = pd.to_datetime(weather_h["timestamp"], utc=True)\n        weather_h["date"] = weather_h["timestamp"].dt.date\n        daily_list = []\n        for date_val, grp in weather_h.groupby("date"):\n            daily_list.append({"date": pd.to_datetime(date_val), "t_mean": grp["temperature"].mean(), "t_max": grp["temperature"].max(), "t_min": grp["temperature"].min(), "rh_mean": grp["relative_humidity"].mean(), "hours_above_90_rh": int((grp["relative_humidity"] >= 90.0).sum())})\n        weather_d = pd.DataFrame(daily_list)\n    else:\n        weather_h = pd.DataFrame(columns=["timestamp", "temperature", "relative_humidity"])\n        weather_d = pd.DataFrame(columns=["date", "t_mean", "t_max", "t_min", "rh_mean"])\n    write_parquet(weather_h, processed_dir / "weather_hourly.parquet")\n    write_parquet(weather_d, processed_dir / "weather_daily.parquet")\n    image_records = []\n    first_frame_ref_area = {}\n    sample_mask_paths = {}\n    for mask_path in sorted(ts_dir.rglob("*_mask.png")):\n        sample_id = mask_path.parent.name\n        sample_mask_paths.setdefault(sample_id, []).append(mask_path)\n    for sample_id, mask_list in sample_mask_paths.items():\n        if not mask_list: continue\n        try:\n            arr = np.array(Image.open(mask_list[0]))\n            m = compute_mask_metrics(arr)\n            first_frame_ref_area[sample_id] = max(m["leaf_pixels_frame"], 1)\n        except Exception:\n            first_frame_ref_area[sample_id] = 10000\n    sequence_records = []\n    for sample_id, mask_list in sample_mask_paths.items():\n        ref_area = first_frame_ref_area[sample_id]\n        plot_id = sample_id.split("_")[0]\n        plot_meta = exp_df[exp_df["plot_id"] == plot_id]\n        if not plot_meta.empty:\n            cultivar, treatment, batch, year = plot_meta.iloc[0]["cultivar"], plot_meta.iloc[0]["treatment"], plot_meta.iloc[0]["batch"], plot_meta.iloc[0]["year"]\n        else:\n            cultivar, treatment, batch, year = "Unknown", "Untreated", 1, 2023\n        sample_images = []\n        first_ts = None\n        for mask_path in mask_list:\n            stem = mask_path.name.replace("_mask.png", "")\n            meta = parse_image_id(stem)\n            if first_ts is None: first_ts = meta["timestamp"]\n            hours_since_first = (meta["timestamp"] - first_ts).total_seconds() / 3600.0\n            arr = np.array(Image.open(mask_path))\n            m = compute_mask_metrics(arr)\n            severity = float(np.clip(m["diseased_pixels"] / float(ref_area), 0.0, 1.0))\n            rec = {"image_id": meta["image_id"], "sample_id": sample_id, "plot_id": plot_id, "leaf_no": meta["leaf_no"], "cultivar": cultivar, "treatment": treatment, "batch": batch, "year": year, "timestamp": meta["timestamp"], "hours_since_first": round(hours_since_first, 2), "severity": severity, "severity_per_frame": float(np.clip(m["diseased_pixels"] / float(max(m["leaf_pixels_frame"], 1)), 0.0, 1.0)), "necrosis_fraction": float(m["necrosis_pixels"] / float(ref_area)), "pustule_density": float(m["pustule_pixels"] / float(ref_area)), "n_lesions": m["n_lesions"], "mean_lesion_area": m["mean_lesion_area"], "diseased_pixels": m["diseased_pixels"], "reference_leaf_area": ref_area, "missing_alignment": False, "washed_off_markers": False, "missing_masks": False}\n            image_records.append(rec)\n            sample_images.append(rec)\n        if sample_images:\n            span_days = (sample_images[-1]["timestamp"] - sample_images[0]["timestamp"]).total_seconds() / 86400.0\n            sequence_records.append({"sample_id": sample_id, "n_images": len(sample_images), "span_days": round(span_days, 2), "cultivar": cultivar, "plot_id": plot_id, "treatment": treatment, "batch": batch, "year": year, "dominant_disease": "Septoria / Rust"})\n    images_df, sequences_df = pd.DataFrame(image_records), pd.DataFrame(sequence_records)\n    write_parquet(images_df, processed_dir / "images_long.parquet")\n    write_parquet(sequences_df, processed_dir / "processed_sequences.parquet")\n    write_parquet(sequences_df, processed_dir / "sequences.parquet")\n    return {"images_count": len(images_df), "sequences_count": len(sequences_df), "weather_days": len(weather_d)}\n''')\n
    # Write ml/data/make_pairs.py\n    with open('/content/ml/data/make_pairs.py', 'w') as f:\n        f.write('''import sys, argparse\nfrom pathlib import Path\nfrom typing import Any, Optional\nimport numpy as np, pandas as pd\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import read_parquet, write_parquet\nlogger = get_logger(__name__)\n\ndef generate_pairs_from_images(images_df: pd.DataFrame, weather_d_df: pd.DataFrame, min_dt_days: float = 1.0, max_dt_days: float = 10.0) -> pd.DataFrame:\n    if images_df.empty: return pd.DataFrame()\n    pair_records = []\n    df = images_df.copy()\n    df["timestamp"] = pd.to_datetime(df["timestamp"])\n    df["date"] = df["timestamp"].dt.date\n    for sample_id, group in df.groupby("sample_id"):\n        sorted_imgs = group.sort_values("timestamp").to_dict("records")\n        n_imgs = len(sorted_imgs)\n        sample_pairs = []\n        for i in range(n_imgs):\n            rec_i = sorted_imgs[i]\n            ts_i = rec_i["timestamp"]\n            for j in range(i + 1, n_imgs):\n                rec_j = sorted_imgs[j]\n                ts_j = rec_j["timestamp"]\n                dt_days = (ts_j - ts_i).total_seconds() / 86400.0\n                if min_dt_days <= dt_days <= max_dt_days:\n                    date_i, date_j = ts_i.date(), ts_j.date()\n                    if not weather_d_df.empty and "date" in weather_d_df.columns:\n                        w_sub = weather_d_df[(pd.to_datetime(weather_d_df["date"]).dt.date >= date_i) & (pd.to_datetime(weather_d_df["date"]).dt.date <= date_j)]\n                        t_mean_int = float(w_sub["t_mean"].mean()) if not w_sub.empty else 18.0\n                        rh_mean_int = float(w_sub["rh_mean"].mean()) if not w_sub.empty else 75.0\n                    else: t_mean_int, rh_mean_int = 18.0, 75.0\n                    sample_pairs.append({"sample_id": sample_id, "image_id_i": rec_i["image_id"], "image_id_j": rec_j["image_id"], "timestamp_i": ts_i, "timestamp_j": ts_j, "dt_days": round(dt_days, 3), "severity_i": rec_i["severity"], "severity_j": rec_j["severity"], "n_lesions_i": rec_i.get("n_lesions", 0), "mean_lesion_area_i": rec_i.get("mean_lesion_area", 0.0), "t_mean_interval": round(t_mean_int, 2), "rh_mean_interval": round(rh_mean_int, 2), "cultivar": rec_i["cultivar"], "treatment": rec_i["treatment"], "batch": rec_i["batch"], "year": rec_i["year"]})\n        weight = 1.0 / max(len(sample_pairs), 1)\n        for pair in sample_pairs:\n            pair["sample_weight"] = round(weight, 5)\n            pair_records.append(pair)\n    return pd.DataFrame(pair_records)\n\ndef process_pairs(processed_dir: Optional[Path] = None) -> dict:\n    paths = get_paths()\n    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "eth"\n    images_path = processed_dir / "images_long.parquet"\n    weather_path = processed_dir / "weather_daily.parquet"\n    if not images_path.exists():\n        from ml.data.eth_wheat import process_eth_dataset\n        process_eth_dataset(processed_dir=processed_dir)\n    images_df = read_parquet(images_path)\n    weather_d_df = read_parquet(weather_path) if weather_path.exists() else pd.DataFrame()\n    pairs_df = generate_pairs_from_images(images_df, weather_d_df)\n    out_path = processed_dir / "pairs.parquet"\n    write_parquet(pairs_df, out_path)\n    pairs_3d = pairs_df[(pairs_df["dt_days"] >= 2.0) & (pairs_df["dt_days"] <= 4.0)] if not pairs_df.empty else pd.DataFrame()\n    pairs_7d = pairs_df[(pairs_df["dt_days"] >= 5.0) & (pairs_df["dt_days"] <= 9.0)] if not pairs_df.empty else pd.DataFrame()\n    return {"total_pairs": len(pairs_df), "pairs_3d_window": len(pairs_3d), "pairs_7d_window": len(pairs_7d)}\n''')\n
    # Write ml/data/splits.py\n    with open('/content/ml/data/splits.py', 'w') as f:\n        f.write('''import sys, argparse\nfrom pathlib import Path\nfrom typing import Any, Optional\nimport numpy as np, pandas as pd\nfrom sklearn.model_selection import GroupKFold\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import read_parquet, write_json\nlogger = get_logger(__name__)\n\ndef create_leak_free_splits(sequences_df: pd.DataFrame, n_folds: int = 5, cal_fraction: float = 0.20, seed: int = 42) -> dict:\n    if sequences_df.empty: return {"leave_cultivar_out": [], "leave_batch_out": [], "leave_year_out": []}\n    np.random.seed(seed)\n    df = sequences_df.copy()\n    unique_samples = df["sample_id"].unique().tolist()\n    splits_dict = {"metadata": {"total_samples": len(unique_samples), "n_folds": n_folds, "cal_fraction": cal_fraction, "seed": seed}, "leave_cultivar_out": [], "leave_batch_out": [], "leave_year_out": []}\n    for grp_col, key in [("cultivar", "leave_cultivar_out"), ("batch", "leave_batch_out"), ("year", "leave_year_out")]:\n        if grp_col in df.columns:\n            groups = df[grp_col].unique()\n            if len(groups) >= 2:\n                n_splits_actual = min(n_folds, len(groups))\n                gkf = GroupKFold(n_splits=n_splits_actual)\n                for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df[grp_col])):\n                    train_samples = df.iloc[train_idx]["sample_id"].unique().tolist()\n                    test_samples = df.iloc[test_idx]["sample_id"].unique().tolist()\n                    n_cal = int(len(train_samples) * cal_fraction)\n                    np.random.shuffle(train_samples)\n                    cal_samples, fit_train_samples = train_samples[:n_cal], train_samples[n_cal:]\n                    grp_val = df.iloc[test_idx][grp_col].iloc[0]\n                    splits_dict[key].append({"fold": fold_idx, "held_out_group": int(grp_val) if isinstance(grp_val, (int, np.integer)) else str(grp_val), "train_samples": fit_train_samples, "calibration_samples": cal_samples, "test_samples": test_samples})\n            else:\n                logger.warning(f"Skipping {key} because len(groups)={len(groups)} < 2")\n    return splits_dict\n\ndef verify_zero_leakage(splits_dict: dict) -> bool:\n    for key in ["leave_cultivar_out", "leave_batch_out", "leave_year_out"]:\n        for fold in splits_dict.get(key, []):\n            tr, cal, te = set(fold["train_samples"]), set(fold["calibration_samples"]), set(fold["test_samples"])\n            if tr.intersection(cal) or tr.intersection(te) or cal.intersection(te): return False\n    return True\n\ndef process_splits(processed_dir: Optional[Path] = None) -> dict:\n    paths = get_paths()\n    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "eth"\n    seq_path = processed_dir / "sequences.parquet"\n    if not seq_path.exists():\n        from ml.data.eth_wheat import process_eth_dataset\n        process_eth_dataset(processed_dir=processed_dir)\n    sequences_df = read_parquet(seq_path)\n    splits_dict = create_leak_free_splits(sequences_df)\n    if not verify_zero_leakage(splits_dict): raise ValueError("Data leakage assertion failed during split generation!")\n    out_path = processed_dir / "splits.json"\n    write_json(splits_dict, out_path)\n    return splits_dict\n''')\n
    # Write ml/data/eda_eth.py\n    with open('/content/ml/data/eda_eth.py', 'w') as f:\n        f.write('''import sys, argparse\nfrom pathlib import Path\nfrom typing import Any, Optional\nimport numpy as np, pandas as pd, matplotlib.pyplot as plt\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import read_parquet\nlogger = get_logger(__name__)\n\ndef run_eth_eda(processed_dir: Optional[Path] = None, figures_dir: Optional[Path] = None, tables_dir: Optional[Path] = None) -> dict:\n    paths = get_paths()\n    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "eth"\n    if figures_dir is None: figures_dir = paths.figures_dir\n    if tables_dir is None: tables_dir = paths.tables_dir\n    figures_dir.mkdir(parents=True, exist_ok=True)\n    tables_dir.mkdir(parents=True, exist_ok=True)\n    images_path = processed_dir / "images_long.parquet"\n    sequences_path = processed_dir / "sequences.parquet"\n    if not images_path.exists():\n        from ml.data.eth_wheat import process_eth_dataset\n        process_eth_dataset(processed_dir=processed_dir)\n    images_df = read_parquet(images_path)\n    sequences_df = read_parquet(sequences_path) if sequences_path.exists() else pd.DataFrame()\n    results = {"figures": [], "tables": []}\n    if "treatment" in images_df.columns and "hours_since_first" in images_df.columns:\n        plt.figure(figsize=(8, 5))\n        for trt, grp in images_df.groupby("treatment"):\n            for sample_id, s_grp in grp.groupby("sample_id"):\n                plt.plot(s_grp["hours_since_first"] / 24.0, s_grp["severity"], alpha=0.3, linewidth=1)\n        plt.xlabel("Elapsed Time (Days)")\n        plt.ylabel("Disease Severity (Fraction)")\n        plt.title("ETH Wheat Disease Severity Trajectories by Treatment")\n        plt.grid(True, linestyle=":", alpha=0.6)\n        plt.tight_layout()\n        fig_path = figures_dir / "eth_severity_trajectories_by_treatment.png"\n        plt.savefig(fig_path, dpi=200)\n        plt.close()\n        results["figures"].append(str(fig_path))\n    if not sequences_df.empty:\n        trt_summary = sequences_df.groupby("treatment").agg(n_sequences=("sample_id", "count"), avg_n_images=("n_images", "mean"), avg_span_days=("span_days", "mean")).reset_index()\n        trt_summary_path = tables_dir / "eth_treatment_summary.csv"\n        trt_summary.to_csv(trt_summary_path, index=False)\n        results["tables"].append(str(trt_summary_path))\n    return results\n''')\n    print("Inline pipeline modules written successfully to /content/ml/data/")

In [ ]:
# Step 3: Run ETH wheat metadata and mask processing
import sys, importlib
if 'ml.data.eth_wheat' in sys.modules:
    import ml.data.eth_wheat
    importlib.reload(ml.data.eth_wheat)
from ml.data.eth_wheat import process_eth_dataset
res_eth = process_eth_dataset()
print('ETH Dataset Processing Summary:')
print(res_eth)

In [ ]:
# Step 4: Generate sequence image pairs for forecasting
import sys, importlib
if 'ml.data.make_pairs' in sys.modules:
    import ml.data.make_pairs
    importlib.reload(ml.data.make_pairs)
from ml.data.make_pairs import process_pairs
res_pairs = process_pairs()
print('Pair Generation Summary:')
print(res_pairs)

In [ ]:
# Step 5: Build leak-free cross-validation splits
import sys, importlib
if 'ml.data.splits' in sys.modules:
    import ml.data.splits
    importlib.reload(ml.data.splits)
from ml.data.splits import process_splits
res_splits = process_splits()
print('Split Generation Complete. Metadata:')
print(res_splits.get('metadata'))

In [ ]:
# Step 6: Generate EDA figures and tables
import sys, importlib
if 'ml.data.eda_eth' in sys.modules:
    import ml.data.eda_eth
    importlib.reload(ml.data.eda_eth)
from ml.data.eda_eth import run_eth_eda
res_eda = run_eth_eda()
print('Generated Figures:', res_eda['figures'])
print('Generated Tables:', res_eda['tables'])

In [ ]:
# Step 7: Export & Download Reports Zip
import shutil
from google.colab import files
from pathlib import Path

reports_dir = Path('/content/reports')
if reports_dir.exists():
    print('Compressing reports directory into /content/phase02_reports.zip...')
    shutil.make_archive('/content/phase02_reports', 'zip', str(reports_dir))
    print('Triggering browser download for phase02_reports.zip...')
    files.download('/content/phase02_reports.zip')
else:
    print('Reports directory /content/reports not found. Please run Step 6 first.')